# 01: Exploratory Data Analysis (EDA)

**Objective:** Safe exploration to understand distributions and drift without altering data.

**Key Actions:**
- Verify Sequence Lengths (Confirm 86 weeks per HCP).
- Distributional Analysis between Labeled/Unlabeled cohorts (Calculate Population Stability Index - PSI).
- Visualize prescription asymmetries and spatial outliers.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)
sns.set_theme(style="whitegrid")

# silver_df = pd.read_parquet('data/silver/silver_layer.parquet')

## 1. Sequence Length Verification

In [ ]:
def verify_sequence_lengths(df: pd.DataFrame, id_col: str = 'NUEVO_ID', expected_weeks: int = 86):
    """
    Confirms that all entities contain exactly the required number of temporal observations.
    """
    if df.empty: return
    
    seq_counts = df.groupby(id_col).size()
    invalid_counts = seq_counts[seq_counts != expected_weeks]
    
    if not invalid_counts.empty:
        logger.warning(f"{len(invalid_counts)} entities possess an invalid sequence length.")
    else:
        logger.info(f"Success: All entities have exactly {expected_weeks} weeks.")

# verify_sequence_lengths(silver_df)

## 2. Population Stability Index (PSI)
Compare labeled (`ATSEG`) vs unlabeled covariate shifts.

In [ ]:
def calculate_psi(expected_array: np.ndarray, actual_array: np.ndarray, buckets: int = 10) -> float:
    """
    Calculates the Population Stability Index for a single continuous feature.
    """
    breakpoints = np.arange(0, buckets + 1) / buckets * 100
    breakpoints = np.percentile(expected_array, breakpoints)
    breakpoints[0] = -np.inf # Catch all lower values
    breakpoints[-1] = np.inf # Catch all upper values
    
    expected_percents = np.histogram(expected_array, breakpoints)[0] / len(expected_array)
    actual_percents = np.histogram(actual_array, breakpoints)[0] / len(actual_array)
    
    def sub_psi(e, a):
        if a == 0: a = 0.0001
        if e == 0: e = 0.0001
        return (e - a) * np.log(e / a)
    
    psi_value = np.sum(sub_psi(expected_percents[i], actual_percents[i]) for i in range(len(expected_percents)))
    return psi_value

def analyze_cohort_drift(df: pd.DataFrame, feature_col: str, target_col: str = 'ATSEG'):
    """
    Compares drift between labeled and unlabeled entities.
    """
    if df.empty: return
    
    labeled = df[df[target_col].notna()][feature_col].dropna().values
    unlabeled = df[df[target_col].isna()][feature_col].dropna().values
    
    if len(labeled) > 0 and len(unlabeled) > 0:
        psi = calculate_psi(labeled, unlabeled)
        logger.info(f"PSI for {feature_col}: {psi:.4f}")

# analyze_cohort_drift(silver_df, 'TRX_WEEKLY')

## 3. Visual Asymmetry and Outliers

In [ ]:
def plot_prescription_asymmetry(df: pd.DataFrame, target_col: str = 'ATSEG', feature_col: str = 'TRX_WEEKLY'):
    """
    Plots distributions to locate spatial outliers.
    """
    if df.empty: return
    
    df['Cohort'] = np.where(df[target_col].notna(), 'Labeled', 'Unlabeled')
    
    plt.figure(figsize=(10, 6))
    sns.boxplot(data=df, x='Cohort', y=feature_col)
    plt.title(f"Distribution of {feature_col} across Cohorts")
    plt.yscale('log') # often necessary for pharma rx data
    plt.show()

# plot_prescription_asymmetry(silver_df)